Use the integer label mask for cropping. That means the uint16 label image you saved as _masks.tiff (preferred) or _masks.png. Overlays and binary previews are only for QA; do not drive cropping from them.

Below is a clean, robust cropper that:
	•	Prefers TIFF masks, falls back to PNG.
	•	Fixes the basename issue by swapping _yellow to each target channel.
	•	Lets you filter tiny labels.
	•	Crops fixed squares or tight bboxes with optional padding.
	•	Writes per-cell crops for all channels plus a CSV manifest with ROI metadata.

In [ ]:
# %% [markdown]
# # Notebook 2 — Crop per-cell patches from Cellpose masks (yellow → all channels)
# Uses label masks (_masks.tiff preferred, else _masks.png) to crop synchronized patches.

# %%
# Cell 1 — Imports and parameters
from pathlib import Path
from typing import List, Optional, Tuple
import numpy as np
import pandas as pd
from skimage import io, measure
from skimage.transform import resize
from tqdm import tqdm
import shutil

# Project root
project_root = Path("/Users/ashi/github/cm4ai_codefest2025")

# Input raw images: data/<channel>/*_<channel>.jpg
img_root = project_root / "data"

# Where Notebook 1 wrote masks (we handle both names, pick the one that exists)
_masks_candidates = [
    project_root / "analysis" / "cellpose_results2",
    project_root / "analysis" / "cellpose_results",
]

# Output crops and CSV
output_root = project_root / "analysis" / "cell_crops"

# Channels + reference channel driving ROIs
channels: List[str] = ["red", "yellow", "blue", "green"]
ref_channel = "yellow"

# Preferred mask artifact
prefer_tiff = True  # use *_masks.tiff if available, else *_masks.png

# Cropping behavior
crop_size: Optional[int] = 640   # None → tight bbox; else fixed squares
bbox_pad_px: int = 0             # used only when crop_size is None
min_area_px: int = 300           # drop very small labels
max_cells_per_image: Optional[int] = None  # set e.g. 200 to cap output

# Discovery
image_exts = [".jpg", ".jpeg", ".tif", ".tiff", ".png"]

# CSV
csv_name = "pred_cell.csv"
csv_columns = [
    "r_image","y_image","b_image","g_image",
    "output_folder","output_prefix","cell_id","image_id",
    "area_px","centroid_r","centroid_c","bbox_minr","bbox_minc","bbox_maxr","bbox_maxc"
]

In [ ]:
# %%
# Cell 2 — Helpers
def ensure_fresh_output(root: Path, chans: List[str]):
    if root.exists():
        shutil.rmtree(root)
    for ch in chans:
        (root / ch).mkdir(parents=True, exist_ok=True)

def choose_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

def discover_ref_dir(masks_root: Path, ref_channel: str) -> Path:
    # typical layout from Notebook 1
    candidates = [
        masks_root / ref_channel / "png",
        masks_root / ref_channel,              # fallback if not nested
    ]
    d = choose_existing(candidates)
    if d is None:
        raise FileNotFoundError(f"No mask dir found under {masks_root}/{ref_channel}")
    return d

def list_ref_masks(ref_dir: Path, prefer_tiff: bool=True):
    tifs = sorted(list(ref_dir.glob("*_masks.tif"))) + sorted(list(ref_dir.glob("*_masks.tiff")))
    pngs = sorted(list(ref_dir.glob("*_masks.png")))
    if prefer_tiff and tifs:
        return tifs
    if pngs:
        return pngs
    # last resort: recursive search
    tifs_r = sorted(list(ref_dir.rglob("*_masks.tif"))) + sorted(list(ref_dir.rglob("*_masks.tiff")))
    pngs_r = sorted(list(ref_dir.rglob("*_masks.png")))
    if prefer_tiff and tifs_r:
        return tifs_r
    if pngs_r:
        return pngs_r
    return []

def read_label_mask(path: Path) -> np.ndarray:
    m = io.imread(path)
    if m.ndim > 2:
        m = m[..., 0]
    return m

def to_labeled(mask: np.ndarray, min_area: int) -> np.ndarray:
    # accept labeled (uint16 with values >1) or binary; enforce min area
    m = mask[..., 0] if mask.ndim > 2 else mask
    if (m.dtype.kind in "iu") and (m.max() > 1):
        lab = m.astype(np.int32)
    else:
        lab = measure.label(m > 0, connectivity=1)
    if min_area > 0 and lab.max() > 0:
        keep = [r.label for r in measure.regionprops(lab) if r.area >= min_area]
        if keep:
            filt = np.isin(lab, keep)
            lab = measure.label(filt, connectivity=1).astype(np.int32)
        else:
            lab = np.zeros_like(lab, dtype=np.int32)
    return lab

def square_window(center_r: float, center_c: float, size: int, shape_hw: Tuple[int,int]) -> Tuple[int,int,int,int]:
    H, W = shape_hw
    half = size // 2
    r0 = int(round(center_r)) - half
    c0 = int(round(center_c)) - half
    r0 = max(r0, 0); c0 = max(c0, 0)
    r1 = min(r0 + size, H)
    c1 = min(c0 + size, W)
    if r1 - r0 < size: r0 = max(r1 - size, 0)
    if c1 - c0 < size: c0 = max(c1 - size, 0)
    return (r0, c0, r1, c1)

def bbox_with_pad(bbox, pad: int, shape_hw: Tuple[int,int]) -> Tuple[int,int,int,int]:
    minr, minc, maxr, maxc = bbox
    r0 = max(minr - pad, 0)
    c0 = max(minc - pad, 0)
    r1 = min(maxr + pad, shape_hw[0])
    c1 = min(maxc + pad, shape_hw[1])
    return (r0, c0, r1, c1)

def safe_crop(img: np.ndarray, win: Tuple[int,int,int,int], pad_value=0) -> np.ndarray:
    r0, c0, r1, c1 = win
    H, W = img.shape[:2]
    sr0, sc0 = max(r0, 0), max(c0, 0)
    sr1, sc1 = min(r1, H), min(c1, W)
    out_h, out_w = r1 - r0, c1 - c0
    out_shape = (out_h, out_w) if img.ndim == 2 else (out_h, out_w, img.shape[2])
    out = np.full(out_shape, pad_value, dtype=img.dtype)
    dr, dc = sr0 - r0, sc0 - c0
    out[dr:dr + (sr1 - sr0), dc:dc + (sc1 - sc0)] = img[sr0:sr1, sc0:sc1]
    return out

def swap_suffix_yellow_to_channel(stem_with_channel: str, target_ch: str, ref_ch: str) -> str:
    # convert "..._yellow" -> "..._<target_ch>"
    if stem_with_channel.endswith(f"_{ref_ch}"):
        return stem_with_channel[:-(len(ref_ch)+1)] + f"_{target_ch}"
    return stem_with_channel  # fallback

def read_image_for_channel(stem_yellow: str, ch: str, ch_dir: Path):
    target_stem = swap_suffix_yellow_to_channel(stem_yellow, ch, ref_channel)
    for ext in image_exts:
        p = ch_dir / f"{target_stem}{ext}"
        if p.exists():
            return io.imread(p), p
    return None, None

In [ ]:
# %%
# Cell 3 — Locate masks robustly
masks_root = choose_existing(_masks_candidates)
if masks_root is None:
    raise FileNotFoundError(f"Neither { _masks_candidates[0] } nor { _masks_candidates[1] } exists.")

ref_dir = discover_ref_dir(masks_root, ref_channel)
mask_files = list_ref_masks(ref_dir, prefer_tiff=prefer_tiff)
if not mask_files:
    raise FileNotFoundError(f"No *_masks.tif(f)/png found under {ref_dir}. Run Notebook 1 for {ref_channel} first.")

print(f"Using masks_root = {masks_root}")
print(f"Reference dir   = {ref_dir}")
print(f"Found {len(mask_files)} reference masks")

In [ ]:
# %%
# Cell 4 — Crop per-cell patches across all channels
ensure_fresh_output(output_root, channels)
rows = []
total_regions = 0

for mpath in tqdm(mask_files, desc="Cropping from reference masks"):
    # e.g. stem_with_suffix = "..._z01_yellow"
    stem_with_suffix = mpath.stem.replace("_masks", "")

    # Read label mask and filter
    lab = to_labeled(read_label_mask(mpath), min_area=min_area_px)
    regions = measure.regionprops(lab)
    if not regions:
        continue

    if max_cells_per_image is not None and len(regions) > max_cells_per_image:
        regions = sorted(regions, key=lambda r: r.area, reverse=True)[:max_cells_per_image]

    # Load raw images once per channel
    raw_by_ch = {}
    for ch in channels:
        ch_dir = img_root / ch
        img, img_path = read_image_for_channel(stem_with_suffix, ch, ch_dir)
        raw_by_ch[ch] = (img, img_path)

    for i, reg in enumerate(regions, start=1):
        if crop_size is not None:
            win = square_window(reg.centroid[0], reg.centroid[1], crop_size, lab.shape)
        else:
            win = bbox_with_pad(reg.bbox, bbox_pad_px, lab.shape)

        out_paths = {}
        for ch in channels:
            img, img_path = raw_by_ch[ch]
            if img is None:
                continue
            crop = safe_crop(img, win)
            if crop_size is not None and (crop.shape[0] != crop_size or crop.shape[1] != crop_size):
                crop = resize(crop, (crop_size, crop_size), order=1, mode="edge",
                              anti_aliasing=True, preserve_range=True).astype(img.dtype)

            out_p = output_root / ch / f"{stem_with_suffix}_cell{i}.png"
            io.imsave(out_p, crop, check_contrast=False)
            out_paths[ch] = str(out_p)

        rows.append([
            out_paths.get("red",""),
            out_paths.get("yellow",""),
            out_paths.get("blue",""),
            out_paths.get("green",""),
            str(output_root),
            f"{stem_with_suffix}_",
            str(i),
            stem_with_suffix,
            int(reg.area),
            float(reg.centroid[0]), float(reg.centroid[1]),
            int(reg.bbox[0]), int(reg.bbox[1]), int(reg.bbox[2]), int(reg.bbox[3]),
        ])
        total_regions += 1

df = pd.DataFrame(rows, columns=csv_columns)
csv_path = output_root / csv_name
df.to_csv(csv_path, index=False)
print(f"Wrote {len(df)} rows to {csv_path}")
print(f"Total crops written: {total_regions}")

In [ ]:
# %%
# Cell 5 — Post-run checks
counts = {ch: len(list((output_root / ch).glob("*.png"))) for ch in channels if (output_root / ch).exists()}
print("Crop counts per channel:", counts)
if len(counts) == len(channels) and len(set(counts.values())) == 1:
    print("Counts match across channels. Synchronized cropping succeeded.")
else:
    print("Counts differ. Check for missing raw images, stem mismatches, or empty masks.")

Summary
	•	Use _masks.tiff (or _masks.png if TIFF missing) as the single source of truth for cropping.
	•	Keep segmentation and cropping in separate notebooks (Notebook 1 = segmentation; Notebook 2 = cropper above).
	•	The code above fixes the folder mismatch and is resilient to both cellpose_results2 and cellpose_results layouts.